In [1]:
#통계 검증 예제1
# (1) 타이타닉호 데이터
#      - 타이타닉호 티켓 가격에 따른 생존률이 궁금하다.
#      -> 생존자 그룹과 비생존자 그룹 간, 타이타닉호 탑승권의 가격은 달랐을 것이다.
#
#      Ho : 생존자 그룹의 타이타닉호 평균 탑승권 가격 == 비생존자 그룹의 타이타닉호 평균 탑승권 가격
#      Ha : 생존자 그룹의 타이타닉호 평균 탑승권 가격 > 비생존자 그룹의 타이타닉호 평균 탑승권 가격

In [1]:
getwd()

[1] "/content"

In [2]:
list.files()

[1] "sample_data"        "Titanic_exdata.csv"

In [7]:
# CSV 파일 불러오기
titanic_df <- read.csv('/content/Titanic_exdata.csv')

# 데이터프레임의 첫 5행 표시
cat("불러온 데이터의 첫 5행:\n")
print(head(titanic_df))

불러온 데이터의 첫 5행:
  PassengerId Survived Pclass
1           1        0      3
2           2        1      1
3           3        1      3
4           4        1      1
5           5        0      3
6           6        0      3
                                                 Name    Sex Age SibSp Parch
1                             Braund, Mr. Owen Harris   male  22     1     0
2 Cumings, Mrs. John Bradley (Florence Briggs Thayer) female  38     1     0
3                              Heikkinen, Miss. Laina female  26     0     0
4        Futrelle, Mrs. Jacques Heath (Lily May Peel) female  35     1     0
5                            Allen, Mr. William Henry   male  35     0     0
6                                    Moran, Mr. James   male  NA     0     0
            Ticket    Fare Cabin Embarked
1        A/5 21171  7.2500              S
2         PC 17599 71.2833   C85        C
3 STON/O2. 3101282  7.9250              S
4           113803 53.1000  C123        S
5           373450  8.0500

이제 `Fare` (요금) 데이터를 생존 여부(Survived)에 따라 두 그룹으로 분리하고, 단측 t-검정을 수행하겠습니다.

In [8]:
# 생존자 그룹과 비생존자 그룹으로 분리
# R에서는 subset을 사용하여 조건을 만족하는 데이터를 추출할 수 있습니다.
survivors_fare <- subset(titanic_df, Survived == 1)$Fare
non_survivors_fare <- subset(titanic_df, Survived == 0)$Fare

# 결측값 제거 (t.test는 NA를 자동으로 처리하지 않으므로 명시적으로 제거)
survivors_fare <- na.omit(survivors_fare)
non_survivors_fare <- na.omit(non_survivors_fare)

# 독립 표본 t-검정 수행 (단측 검정)
# Ha : 생존자 그룹의 타이타닉호 평균 탑승권 가격 > 비생존자 그룹의 타이타닉호 평균 탑승권 가격
# alternative = "greater"를 사용하여 단측 검정 수행
# var.equal = FALSE는 Welch's t-test를 의미합니다.
t_test_result <- t.test(survivors_fare, non_survivors_fare, alternative = "greater", var.equal = FALSE)

# 결과 추출
t_statistic <- t_test_result$statistic
one_tailed_p_value <- t_test_result$p.value

mean_survivors <- mean(survivors_fare)
mean_non_survivors <- mean(non_survivors_fare)

cat(sprintf("생존자 그룹 평균 Fare: %.2f\n", mean_survivors))
cat(sprintf("비생존자 그룹 평균 Fare: %.2f\n", mean_non_survivors))
cat(sprintf("T-statistic: %.3f\n", t_statistic))
cat(sprintf("단측 p-value: %.3f\n", one_tailed_p_value))

# 결과 해석
alpha <- 0.05 # 유의수준

if (one_tailed_p_value < alpha) {
    cat(sprintf("\n(단측 p-value %.3f < 유의수준 %.2f)\n", one_tailed_p_value, alpha))
    cat("귀무가설(Ho)을 기각합니다. 생존자 그룹의 평균 탑승권 가격이 비생존자 그룹보다 통계적으로 유의하게 높다고 할 수 있습니다.\n")
} else {
    cat(sprintf("\n(단측 p-value %.3f >= 유의수준 %.2f)\n", one_tailed_p_value, alpha))
    cat("귀무가설(Ho)을 기각할 충분한 증거가 없습니다. 생존자 그룹의 평균 탑승권 가격이 비생존자 그룹보다 통계적으로 유의하게 높다고 할 수 없습니다.\n")
}

생존자 그룹 평균 Fare: 48.40
비생존자 그룹 평균 Fare: 22.12
T-statistic: 6.839
단측 p-value: 0.000

(단측 p-value 0.000 < 유의수준 0.05)
귀무가설(Ho)을 기각합니다. 생존자 그룹의 평균 탑승권 가격이 비생존자 그룹보다 통계적으로 유의하게 높다고 할 수 있습니다.


#### **결과 해석 가이드:**

*   **T-statistic**: 이 값이 양수이면 생존자 그룹의 평균 요금이 비생존자 그룹보다 높다는 것을, 음수이면 그 반대임을 의미합니다.
*   **단측 p-value**: 이 값이 미리 설정한 유의수준(일반적으로 0.05)보다 작으면, 귀무가설(Ho: 두 그룹의 평균 요금이 같다)을 기각하고 대립가설(Ha: 생존자 그룹의 평균 요금이 비생존자 그룹보다 높다)을 채택합니다.

이 분석은 생존 여부가 `Fare`에 미치는 영향을 이해하는 데 도움을 줍니다.

In [ ]:
#메뉴얼 t-test
